In [1]:
# Tải mô hình Sam_vit_b_01ec64
!rm -f /content/sam_vit_b_01ec64.pth

!wget -O /content/sam_vit_b_01ec64.pth \
https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

--2026-11-04 08:06:30--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.86.101.115, 99.86.101.39, 99.86.101.47, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.86.101.115|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375042383 (358M) [binary/octet-stream]
Saving to: ‘/content/sam_vit_b_01ec64.pth’

/content/sam_vit_b_ 100%[===================>] 357.67M   268MB/s    in 1.3s    

2026-11-04 08:06:31 (268 MB/s) - ‘/content/sam_vit_b_01ec64.pth’ saved [375042383/375042383]



In [2]:
# Cài đặt unrar nếu chưa có
!apt-get install unrar

# Giải nén file vào thư mục hiện tại (/content/)
!unrar x skin-disease-datasaet.rar /content/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:7.0.7-1build1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.

UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from skin-disease-datasaet.rar

Creating    /content/skin-disease-datasaet                            OK
Creating    /content/skin-disease-datasaet/test_set                   OK
Creating    /content/skin-disease-datasaet/test_set/BA- cellulitis    OK
Extracting  /content/skin-disease-datasaet/test_set/BA- cellulitis/BA- cellulitis (1).webp       0%  OK 
Extracting  /content/skin-disease-datasaet/test_set/BA- cellulitis/BA- cellulitis (104).jpg       0%  OK 
Extracting  /content/skin-disease-datasaet/test_set/BA- cellulitis/BA- cellulitis (106).jpg       0%  OK 
Extracting  /content/skin-disease-datasaet/test_set/BA- cellulitis/BA- cellulitis (115).jpg       0%

In [4]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-cqerjo05
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-cqerjo05
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36637 sha256=3cc765170e8a90bd3c6041c58d4633dd70f8f081b1c3f4072998e57bfedca41a
  Stored in directory: /tmp/pip-ephem-wheel-cache-2j95_vkq/wheels/5d/7b/4a/c7fda89f2e27647dd0816f51ea103992b1b6f0f53ad234895c
Successfully built segment_anything


In [5]:

import os
import cv2
import time
import random
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
from segment_anything import sam_model_registry, SamPredictor

# ================================================================
# CẤU HÌNH
# ================================================================
LABELS = [
    "BA- cellulitis", "BA-impetigo", "FU-athlete-foot", "FU-nail-fungus",
    "FU-ringworm", "PA-cutaneous-larva-migrans", "VI-chickenpox", "VI-shingles"
]

DATA_DIR = '/content/skin-disease-datasaet/train_set' # đường dẫn vào thư mục ảnh train
OUTPUT_MASK_DIR = 'output_pseudo_masks_HSV_SAM' # thư mục chứa ảnh sau khi tạo Masks HSV+SAM
IMAGE_SIZE = 224

SAM_CHECKPOINT = "/content/sam_vit_b_01ec64.pth" # mô hình tạo Masks SAM
MODEL_TYPE = "vit_b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [6]:
# ================================================================
# I. HÀM: TẠO MASK HSV
# ================================================================
# --- BƯỚC 1: TIỀN XỬ LÝ VÀ CHUYỂN ĐỔI ĐỊNH DẠNG ẢNH
def get_hsv_mask(image_pil):
    image_np = np.array(image_pil)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    image_bgr = cv2.resize(image_bgr, (IMAGE_SIZE, IMAGE_SIZE))
# --- BƯỚC 2: CHUYỂN ĐỔI SANG KHÔNG GIAN MÀU HSV ---
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
# --- BƯỚC 3: ĐỊNH NGHĨA CÁC KHOẢNG NGƯỠNG MÀU (THRESHOLD RANGES) ---
    lower_hsv1 = np.array([0, 30, 30])
    upper_hsv1 = np.array([15, 255, 255])
    lower_hsv2 = np.array([160, 20, 30])
    upper_hsv2 = np.array([179, 255, 255])
# --- BƯỚC 4: LỌC MÀU VÀ TẠO MASK NHỊ PHÂN ---
    mask1 = cv2.inRange(hsv, lower_hsv1, upper_hsv1)
    mask2 = cv2.inRange(hsv, lower_hsv2, upper_hsv2)
    mask_hsv = cv2.bitwise_or(mask1, mask2)
# --- BƯỚC 5: XỬ LÝ HÌNH THÁI HỌC (MORPHOLOGICAL OPERATIONS)
    kernel = np.ones((5, 5), np.uint8)
    mask_hsv = cv2.morphologyEx(mask_hsv, cv2.MORPH_OPEN, kernel, iterations=2)
    mask_hsv = cv2.morphologyEx(mask_hsv, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask_hsv = cv2.dilate(mask_hsv, kernel, iterations=1)
# Trả về bức ảnh mặt nạ nhị phân (Mask) hoàn chỉnh để chuyển tiếp sang giai đoạn tinh chỉnh bằng SAM
    return mask_hsv


In [7]:
# ================================================================
# II. HÀM: SAM Refinement
# ================================================================
def refine_with_sam(image_pil, mask_init, predictor):
  # --- BƯỚC 1: CHUẨN BỊ ẢNH ĐẦU VÀO CHO MÔ HÌNH SAM ---
    image_np = np.array(image_pil)
    image_np = cv2.resize(image_np, (IMAGE_SIZE, IMAGE_SIZE))
    predictor.set_image(image_np)
  # --- BƯỚC 2: TRÍCH XUẤT TỌA ĐỘ CÁC ĐIỂM TỪ vùng được phát hiện bởi mask HSV trước đó ---
    input_points = np.argwhere(mask_init > 0)

    if len(input_points) == 0:
        return mask_init
  # --- BƯỚC 3: LẤY MẪU NGẪU NHIÊN CÁC ĐIỂM LÀM ĐẦU VÀO (PROMPTS) CHO SAM ---
    num_points = min(len(input_points), 10)
    sampled_idx = np.random.choice(len(input_points), num_points, replace=False)
    input_point = input_points[sampled_idx][:, [1, 0]]
    input_label = np.ones(num_points)
  # --- BƯỚC 4: THỰC HIỆN DỰ ĐOÁN VÀ PHÂN ĐOẠN VÙNG TỔN THƯƠNG ---
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=False
    )

    if masks is None or len(masks) == 0:
        return mask_init

    return masks[0].astype(np.uint8) * 255


In [8]:
# ================================================================
# III. HÀM: XỬ LÝ TOÀN BỘ ẢNH + BÁO CÁO KẾT QUẢ
# ================================================================
def process_all_images(data_dir, output_mask_dir):
  # --- BƯỚC 1: KHỞI TẠO VÀ KIỂM TRA MÔ HÌNH SAM ---
    print(f"\n--- Bắt đầu tạo Mặt nạ Hybrid (HSV + SAM) ---")
  # Ghi nhận thời gian bắt đầu chạy để đo hiệu năng tổng thể của pipeline
    start_time = time.time()

    os.makedirs(output_mask_dir, exist_ok=True)
    for label in LABELS:
        os.makedirs(os.path.join(output_mask_dir, label), exist_ok=True)
  # Kiểm tra xem file trọng số (checkpoint) của mô hình SAM
    if not os.path.exists(SAM_CHECKPOINT):
        print(f"⚠️ Không tìm thấy checkpoint SAM tại {SAM_CHECKPOINT}")
        return
  # Tải mô hình SAM lên thiết bị (GPU hoặc CPU) và khởi tạo bộ dự đoán (SamPredictor)
    print("Đang tải mô hình SAM...")
    sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT).to(DEVICE)
    predictor = SamPredictor(sam)
    print(f"✅ Đã tải SAM trên thiết bị: {DEVICE}")
  # --- BƯỚC 2: THU THẬP DANH SÁCH TẤT CẢ CÁC FILE ẢNH CẦN XỬ LÝ ---
    image_list = []
    for cls_name in LABELS:
        cls_path = os.path.join(data_dir, cls_name)
        if os.path.isdir(cls_path):
            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_list.append((os.path.join(cls_path, img_name), cls_name, img_name))

    if not image_list:
        print("⚠️ Không có ảnh hợp lệ!")
        return

    print(f"📸 Tổng ảnh cần xử lý: {len(image_list)}")

    total_images = 0
    failed = 0
    sample_results = []  # dùng cho minh họa
  # --- BƯỚC 3: VÒNG LẶP XỬ LÝ TỪNG ẢNH TRONG DATASET
    for img_path, cls_name, img_name in tqdm(image_list, desc="Processing"):
        try:
            image = Image.open(img_path).convert('RGB')
            # 1. Gọi hàm tạo mask bằng phương pháp ngưỡng màu HSV
            mask_hsv = get_hsv_mask(image)
            # 2. Gọi hàm tinh chỉnh mask bằng mô hình (SAM)
            mask_refined = refine_with_sam(image, mask_hsv, predictor)
            # 3. Kết hợp (gộp) mask HSV và mask SAM
            final_mask = cv2.bitwise_or(mask_hsv, mask_refined)

            # lưu mặt nạ
            output_path = os.path.join(output_mask_dir, cls_name, img_name.replace(".jpg", "_mask.png"))
            cv2.imwrite(output_path, final_mask)

            # lưu 5 ví dụ minh họa
            if len(sample_results) < 5:
                sample_results.append((image, mask_hsv, mask_refined, final_mask))

            total_images += 1

        except Exception as e:
            failed += 1
            print(f"⚠️ Lỗi xử lý {img_path}: {e}")

    end_time = time.time()
    runtime = end_time - start_time
    success_rate = (total_images / len(image_list)) * 100
  # --- BƯỚC 4: TỔNG KẾT VÀ IN BÁO CÁO KẾT QUẢ HIỆU NĂNG ---
    print("\n=================📊 BÁO CÁO KẾT QUẢ =================")
    print(f"Tổng ảnh: {len(image_list)}")
    print(f"Thành công: {total_images}")
    print(f"Lỗi xử lý: {failed}")
    print(f"Tỷ lệ tạo mask thành công: {success_rate:.2f}%")
    print(f"Thời gian tổng: {runtime:.2f} giây")
    print(f"Trung bình mỗi ảnh: {runtime/len(image_list):.4f} giây")
    print("=====================================================\n")

    return sample_results
# ================================================================
# CHẠY TRỰC TIẾP TRÊN COLAB
# ================================================================
if __name__ == "__main__":
    process_all_images(DATA_DIR, OUTPUT_MASK_DIR)



--- Bắt đầu tạo Mặt nạ Hybrid (HSV + SAM) ---
Đang tải mô hình SAM...
✅ Đã tải SAM trên thiết bị: cuda
📸 Tổng ảnh cần xử lý: 924


Processing: 100%|██████████| 924/924 [07:27<00:00,  2.07it/s]


=================📊 BÁO CÁO KẾT QUẢ =================
Tổng ảnh: 924
Thành công: 924
Lỗi xử lý: 0
Tỷ lệ tạo mask thành công: 100.00%
Thời gian tổng: 448.47 giây
Trung bình mỗi ảnh: 0.4854 giây

